# Proyecto 2: Campo eléctrico de un anillo cargado y trayectorias de partículas

**Universidad de Los Andes — Física A**

Integrantes: David Bastos y Juan Felipe Tejada

---

## Sección 1: Campo eléctrico del anillo

Un anillo delgado de radio $R$ y carga total $Q>0$ está en el plano $xy$, centrado en el origen. Se modela como $N$ cargas puntuales iguales $q_j = Q/N$ ubicadas en

$$\vec r_j = R\,(\cos\varphi_j,\ \sin\varphi_j,\ 0),\qquad \varphi_j = \frac{2\pi j}{N},\qquad j=0,1,\dots,N-1.$$

Por el principio de superposición, el campo eléctrico en un punto $\vec r$ es (ecuación 2)

$$\vec E(\vec r) = k\sum_{j=0}^{N-1} \frac{q_j\,(\vec r - \vec r_j)}{|\vec r - \vec r_j|^3}.$$

Sobre el eje del anillo se tienen las expresiones analíticas (ecuaciones 3 y 4):

$$E_z(0,0,z) = \frac{k\,Q\,z}{(R^2+z^2)^{3/2}},\qquad V(0,0,z) = \frac{k\,Q}{\sqrt{R^2+z^2}}.$$

Se trabaja en **unidades reducidas** (Tabla 1): $k=1$, $Q=1$, $R=1$, $N=60$, $m=1$, $\Delta t=10^{-3}$.

### Implementación del modelo y del campo

El campo se implementa como una función que recibe un vector posición y devuelve el vector $\vec E$, para reutilizarla en todas las partes del proyecto.

In [ ]:
import numpy as np

# Parámetros comunes (Tabla 1, unidades reducidas)
K = 1.0          # constante de Coulomb
Q = 1.0          # carga total del anillo
R = 1.0          # radio del anillo
N_DEFAULT = 60   # número de cargas del anillo


def posiciones_cargas(N):
    """Posiciones r_j de las N cargas puntuales del anillo (ecuación 1).

    Las cargas se distribuyen uniformemente sobre el círculo de radio R
    en el plano xy. Retorna un arreglo de forma (N, 3).
    """
    phi = 2.0 * np.pi * np.arange(N) / N
    r_j = np.zeros((N, 3))
    r_j[:, 0] = R * np.cos(phi)
    r_j[:, 1] = R * np.sin(phi)
    # r_j[:, 2] = 0  (el anillo está en el plano xy)
    return r_j


def campo_electrico(r, N=N_DEFAULT, r_cargas=None):
    """Campo eléctrico del anillo en el punto r por superposición (ec. 2).

    r        : array_like (3,) punto donde se evalúa el campo.
    N        : número de cargas puntuales del anillo.
    r_cargas : (N, 3) posiciones ya calculadas (opcional, evita recomputar).
    Retorna el vector E de forma (3,).
    """
    r = np.asarray(r, dtype=float)
    if r_cargas is None:
        r_cargas = posiciones_cargas(N)

    q_j = Q / N                       # carga de cada punto
    d = r - r_cargas                  # vectores r - r_j, forma (N, 3)
    dist = np.linalg.norm(d, axis=1)  # |r - r_j|, forma (N,)

    contrib = K * q_j * d / dist[:, None] ** 3
    return contrib.sum(axis=0)


def Ez_analitico(z):
    """Componente E_z sobre el eje del anillo (ecuación 3)."""
    return K * Q * z / (R**2 + z**2) ** 1.5


def potencial_analitico(z):
    """Potencial sobre el eje del anillo (ecuación 4)."""
    return K * Q / np.sqrt(R**2 + z**2)

### Parte 1.1 — Validación de $E_z$ sobre el eje

Se evalúa $E_z$ en los puntos $(0,0,z)$ con $z=0.5,\ 1.0,\ 2.0$ y se compara con la ecuación (3), reportando el error relativo
$$\varepsilon = \frac{|E_z^{\text{num}} - E_z^{\text{exacto}}|}{|E_z^{\text{exacto}}|}.$$

In [ ]:
print(f"{'z':>6} | {'E_z numérico':>14} | {'E_z analítico':>14} | {'err. rel.':>12}")
print("-" * 56)

r_cargas = posiciones_cargas(N_DEFAULT)
for z in (0.5, 1.0, 2.0):
    E = campo_electrico([0.0, 0.0, z], N=N_DEFAULT, r_cargas=r_cargas)
    Ez_num = E[2]
    Ez_exact = Ez_analitico(z)
    err_rel = abs(Ez_num - Ez_exact) / abs(Ez_exact)
    print(f"{z:>6.2f} | {Ez_num:>14.9f} | {Ez_exact:>14.9f} | {err_rel:>12.3e}")

El error relativo es del orden de $10^{-16}$ (precisión de máquina): el campo numérico coincide con la ecuación (3) para los tres valores de $z$. Esto valida la implementación de la ecuación (2).

### Parte 1.2 — Campo en el punto cercano $(0.8,\,0,\,0.1)$ para varios $N$

Se calcula el campo en un punto fuera del eje y cercano al anillo, para $N=10,\ 30,\ 60,\ 120$.

In [ ]:
punto = [0.8, 0.0, 0.1]

print(f"Campo en el punto {tuple(punto)} para distintos N\n")
print(f"{'N':>5} | {'E_x':>12} | {'E_y':>12} | {'E_z':>12} | {'|E|':>12}")
print("-" * 64)

for N in (10, 30, 60, 120):
    E = campo_electrico(punto, N=N)
    magnitud = np.linalg.norm(E)
    print(f"{N:>5} | {E[0]:>12.6f} | {E[1]:>12.6f} | {E[2]:>12.6f} | {magnitud:>12.6f}")

A diferencia del eje, aquí el valor sí depende de $N$: pasa de $|E|\approx 1.86$ con $N=10$ a estabilizarse en $|E|\approx 1.156$ a partir de $N=60$ (los valores de $N=60$ y $N=120$ coinciden hasta la 5.ª–6.ª cifra). Nótese además que $E_y\approx 0$: el punto está en el plano $y=0$, respecto al cual la distribución es simétrica, por lo que la componente $y$ se cancela.

### Parte 1.3 — Escena en vpython: anillo y dos rejillas de flechas

Se construye la escena con:
- Cada una de las $N$ cargas del anillo como una esfera roja de radio $0.04$.
- El campo representado por flechas de **longitud fija** $0.3$ en dirección $\vec E/|\vec E|$, sobre dos rejillas planas: una en $y=0$ (variando $x,z$) y otra en $x=0$ (variando $y,z$), con $x,y,z\in\{-2.0,-1.6,\dots,2.0\}$.
- Se omiten los puntos de la rejilla que estén a menos de $0.15$ de alguna carga del anillo.


In [ ]:
from vpython import canvas, sphere, arrow, vector, color

escena = canvas(title="Proyecto 2 - Campo eléctrico del anillo",
                width=900, height=700, background=color.white)

N = N_DEFAULT
r_cargas = posiciones_cargas(N)

# Anillo: N esferas rojas
for rj in r_cargas:
    sphere(pos=vector(rj[0], rj[1], rj[2]), radius=0.04, color=color.red)

# Rejillas de flechas
valores = np.arange(-2.0, 2.0 + 1e-9, 0.4)  # {-2.0, -1.6, ..., 2.0}
longitud = 0.3                               # longitud fija de cada flecha
dist_min = 0.15                              # distancia mínima a las cargas


def dibujar_flecha(punto):
    """Dibuja una flecha de longitud fija en dirección E/|E| si procede."""
    distancias = np.linalg.norm(punto - r_cargas, axis=1)
    if distancias.min() < dist_min:
        return
    E = campo_electrico(punto, N=N, r_cargas=r_cargas)
    norma = np.linalg.norm(E)
    if norma == 0.0:
        return
    direccion = E / norma
    arrow(pos=vector(punto[0], punto[1], punto[2]),
          axis=vector(direccion[0], direccion[1], direccion[2]) * longitud,
          color=color.blue, shaftwidth=0.02)


# Rejilla en el plano y = 0 (variando x, z)
for x in valores:
    for z in valores:
        dibujar_flecha(np.array([x, 0.0, z]))

# Rejilla en el plano x = 0 (variando y, z)
for y in valores:
    for z in valores:
        dibujar_flecha(np.array([0.0, y, z]))

print("Escena vpython construida.")

---

## Preguntas de la Sección 1

### a. Sobre el eje el acuerdo con la ec. (3) no depende de $N$, pero en $(0.8,0,0.1)$ sí. ¿Por qué? ¿Qué $N$ es suficiente?

**Dependencia de $N$ a partir de la simetría.** Sobre el eje, el punto de evaluación $(0,0,z)$ equidista de las $N$ cargas: todas están a la misma distancia $\sqrt{R^2+z^2}$. Las cargas ocupan posiciones $\varphi_j$ repartidas simétricamente alrededor del círculo, de modo que las componentes transversales ($x,y$) del campo de cada carga se cancelan **exactamente** por pares (o por la simetría rotacional de la distribución discreta), sin importar cuántas cargas haya. Solo sobrevive la suma de las componentes $z$, y como cada carga aporta lo mismo, el resultado es idéntico al de la fórmula continua (ec. 3) para **cualquier** $N\ge 1$. Por eso el error sobre el eje es de precisión de máquina independientemente de $N$.

Fuera del eje, en el punto $(0.8,0,0.1)$, esa equidistancia se rompe: cada carga está a una distancia distinta del punto y la cancelación transversal deja de ser exacta. El arreglo discreto de $N$ cargas solo se aproxima a un anillo continuo de carga cuando la separación angular entre cargas, $2\pi/N$, es pequeña frente a la escala sobre la que varía el integrando. Cerca del anillo ($|\vec r|\approx R$) esa variación es rápida, así que se necesita un $N$ apreciable para que la suma discreta converja al valor continuo.

**Criterio y valor suficiente.** El criterio práctico es aumentar $N$ hasta que el campo deje de cambiar dentro de una tolerancia fijada. En la tabla de la parte 1.2, el paso de $N=60$ a $N=120$ cambia $|E|$ solo en la 5.ª–6.ª cifra significativa (variación relativa $\lesssim 10^{-5}$), mientras que de $N=10$ a $N=30$ el cambio es enorme (de $1.86$ a $1.16$). Por tanto **$N=60$ es suficiente** para este punto, con el criterio de que el campo ya convergió a mejor del $10^{-3}$ relativo respecto a duplicar $N$. Esto justifica el valor $N=60$ de la Tabla 1. (El $N$ requerido crece a medida que el punto se acerca más al anillo.)

### b. ¿Cómo es el campo en el centro, lejos y cerca de las cargas? ¿Relación entre las flechas y las superficies equipotenciales?

**En el centro del anillo** ($\vec r = \vec 0$) el campo es nulo: por la simetría de la distribución, para cada carga existe otra diametralmente opuesta cuyo campo apunta en sentido contrario, y todas las contribuciones se cancelan. En la rejilla las flechas cercanas al origen son muy cortas o ausentes y el campo crece al alejarse del centro dentro del plano del anillo.

**Lejos del anillo** ($|\vec r|\gg R$) la distribución se ve como una carga puntual $Q$ concentrada en el origen: las flechas apuntan radialmente hacia afuera (carga positiva) y su magnitud decae como $1/r^2$. Las flechas se vuelven cada vez más paralelas a la dirección radial $\hat r$.

**Cerca de las cargas** el campo está dominado por la carga puntual más próxima: las flechas apuntan casi radialmente respecto de esa carga y su magnitud crece mucho (diverge al acercarse). Por eso el enunciado pide omitir los puntos de la rejilla a menos de $0.15$ de una carga: ahí el campo es enorme y poco representativo del conjunto.

**Relación con las equipotenciales.** La dirección de las flechas ($\vec E$) es siempre **perpendicular** a las superficies equipotenciales, porque $\vec E = -\nabla V$ y el gradiente es normal a las superficies de $V$ constante. Además $\vec E$ apunta en el sentido de $V$ decreciente (hacia potenciales menores). Así, donde las flechas son radiales (lejos, o cerca de una carga) las equipotenciales son aproximadamente esféricas; en el plano del anillo, cerca del centro, el campo tangente a ese plano indica cómo se curvan las equipotenciales alrededor de la fuente.

### Funciones para las partes 2 y 3




In [ ]:
#parametros extra
M = 1.0            # masa de la partícula
DT_DEFAULT = 1e-3  #Paso del tiempo



def potencial(r, N=N_DEFAULT, r_cargas=None):
    """Potencial del anillo en el punto r por superposición (vale fuera del eje)."""
    r = np.asarray(r, dtype=float)
    if r_cargas is None:
        r_cargas = posiciones_cargas(N)
    dist = np.linalg.norm(r - r_cargas, axis=1)
    return K * (Q / N) * np.sum(1.0 / dist)


def paso(r, v, q, dt, N, r_cargas):
    """Paso de Euler-Cromer"""
    E = campo_electrico(r, N=N, r_cargas=r_cargas)
    v = v + (q * E / M) * dt
    r = r + v * dt
    return r, v


def energias(r, v, q, N, r_cargas):
    """Devuelve (K, U, E_tot) con U = q V(r)."""
    Kc = 0.5 * M * float(v @ v)
    U = q * potencial(r, N=N, r_cargas=r_cargas)
    return Kc, U, Kc + U


def simular(r0, v0, q, tmax, dt=DT_DEFAULT, N=N_DEFAULT, dist_stop=None):
    """Simulación numérica pura (sin gráficos). Retorna un diccionario.

    dist_stop: si no es None, se detiene cuando la distancia a alguna carga
               del anillo sea menor que este valor.
    """
    r_cargas = posiciones_cargas(N)
    r = np.array(r0, dtype=float)
    v = np.array(v0, dtype=float)
    nsteps = int(round(tmax / dt))

    t_l, r_l, v_l, K_l, U_l, E_l = [], [], [], [], [], []
    detenida, t_stop = False, None

    for i in range(nsteps + 1):
        t = i * dt
        Kc, U, Et = energias(r, v, q, N, r_cargas)
        t_l.append(t); r_l.append(r.copy()); v_l.append(v.copy())
        K_l.append(Kc); U_l.append(U); E_l.append(Et)

        if dist_stop is not None:
            if np.linalg.norm(r - r_cargas, axis=1).min() < dist_stop:
                detenida, t_stop = True, t
                break
        r, v = paso(r, v, q, dt, N, r_cargas)

    return dict(t=np.array(t_l), r=np.array(r_l), v=np.array(v_l),
                K=np.array(K_l), U=np.array(U_l), Etot=np.array(E_l),
                detenida=detenida, t_stop=t_stop)


def periodo_desde_z(t, z):
    """Instantes en que z pasa de positivo a negativo (con interpolación lineal)
    y períodos entre cruces consecutivos."""
    idx = np.where((z[:-1] > 0) & (z[1:] <= 0))[0]
    tc = t[idx] + (t[idx + 1] - t[idx]) * z[idx] / (z[idx] - z[idx + 1])
    return tc, np.diff(tc)


# ---------------------------------------------------------------------------
# Visualización con vpython (reutiliza la escena `escena` ya construida)
# ---------------------------------------------------------------------------

from vpython import canvas, graph, gcurve, sphere, arrow, vector, color, rate

_cache_flechas = None


def _flechas_rejilla(N, r_cargas):
    """Calcula una sola vez los pares (punto, dirección E/|E|) de las dos rejillas."""
    global _cache_flechas
    if _cache_flechas is not None and _cache_flechas[0] == N:
        return _cache_flechas[1]
    valores = np.arange(-2.0, 2.0 + 1e-9, 0.4)
    puntos = [np.array([x, 0.0, z]) for x in valores for z in valores]    # plano y = 0
    puntos += [np.array([0.0, y, z]) for y in valores for z in valores]   # plano x = 0
    lista = []
    for p in puntos:
        if np.linalg.norm(p - r_cargas, axis=1).min() < 0.15:
            continue
        E = campo_electrico(p, N=N, r_cargas=r_cargas)
        n = np.linalg.norm(E)
        if n == 0.0:
            continue
        lista.append((p, E / n))
    _cache_flechas = (N, lista)
    return lista


def crear_escena(titulo, N=N_DEFAULT, rango=3.0, con_flechas=True):
    """Canvas con el anillo (esferas rojas) y las dos rejillas de flechas."""
    escena = canvas(title=titulo, width=800, height=600, background=color.white)
    escena.autoscale = False
    escena.range = rango
    r_cargas = posiciones_cargas(N)
    for rj in r_cargas:
        sphere(pos=vector(rj[0], rj[1], rj[2]), radius=0.04, color=color.red)
    if con_flechas:
        for p, d in _flechas_rejilla(N, r_cargas):
            arrow(pos=vector(p[0], p[1], p[2]),
                  axis=vector(d[0], d[1], d[2]) * 0.3,
                  color=color.blue, shaftwidth=0.02)
    return escena


def animar(r0, v0, q, tmax, dt=DT_DEFAULT, N=N_DEFAULT, sobre_eje=True,
           dist_stop=None, cada=10, titulo="", rango=3.0, con_flechas=True):
    """Anima la partícula con rate(500).

    sobre_eje=True  -> panel de energía con K, U y E_tot
    sobre_eje=False -> panel de energía solo con K
    cada            -> cada cuántos pasos se agrega un punto a las gráficas
    rango           -> semiancho de la vista (usa ~6 en la Parte 3)
    con_flechas     -> False para crear la escena más rápido (sin las rejillas)
    """
    crear_escena(titulo, N=N, rango=rango, con_flechas=con_flechas)

    r_cargas = posiciones_cargas(N)
    r = np.array(r0, dtype=float)
    v = np.array(v0, dtype=float)

    col = color.blue if q < 0 else color.orange
    p = sphere(pos=vector(r[0], r[1], r[2]), radius=0.05, color=col,
               make_trail=True, interval=5, trail_radius=0.01)

    g1 = graph(title=f"{titulo} - z(t)", xtitle="t", ytitle="z", width=600, height=250)
    cz = gcurve(graph=g1, color=color.black)
    g2 = graph(title=f"{titulo} - energía", xtitle="t", ytitle="energía", width=600, height=250)
    cK = gcurve(graph=g2, color=color.red, label="K")
    if sobre_eje:
        cU = gcurve(graph=g2, color=color.blue, label="U")
        cE = gcurve(graph=g2, color=color.black, label="E_tot")

    nsteps = int(round(tmax / dt))
    for i in range(nsteps + 1):
        rate(500)
        t = i * dt
        if i % cada == 0:
            Kc, U, Et = energias(r, v, q, N, r_cargas)
            cz.plot(t, r[2])
            cK.plot(t, Kc)
            if sobre_eje:
                cU.plot(t, U)
                cE.plot(t, Et)

        if dist_stop is not None:
            if np.linalg.norm(r - r_cargas, axis=1).min() < dist_stop:
                print(f"Detenida en t = {t:.3f}: distancia a una carga < {dist_stop}")
                break

        r, v = paso(r, v, q, dt, N, r_cargas)
        p.pos = vector(r[0], r[1], r[2])


### Parte 2 Oscilacion de una carga negativa sobre el eje


In [ ]:
# --- Parte 2: carga negativa ---
for z0 in (0.2, 1.5):
    s = simular([0, 0, z0], [0, 0, 0], q=-1, tmax=30)
    tc, T = periodo_desde_z(s["t"], s["r"][:, 2])
    print(f"z0={z0}: T medido = {T.mean():.4f} (T0 = {2*np.pi:.4f})")

# Parte 2.3: energía con dt = 1e-3 y 1e-2
for z0 in (0.2, 1.5):
    for dt in (1e-3, 1e-2):
        s = simular([0, 0, z0], [0, 0, 0], q=-1, tmax=30, dt=dt)
        print(f"z0={z0}, dt={dt}: E_final - E_inicial = {s['Etot'][-1]-s['Etot'][0]:.3e}, "
              f"rango de E_tot = {s['Etot'].max()-s['Etot'].min():.3e}")

# Parte 2.4
s = simular([0.01, 0, 0.2], [0, 0, 0], q=-1, tmax=10, dist_stop=0.05)
print(s["detenida"], s["t_stop"], s["r"][-1])


### 2.1

In [ ]:
animar([0, 0, 0.2],  [0, 0, 0], q=-1, tmax=30, sobre_eje=True,  rango=3, titulo="2.1")

### 2.2

In [ ]:
animar([0, 0, 1.5],  [0, 0, 0], q=-1, tmax=30, sobre_eje=True,  rango=3, titulo="2.2")


### 2.3

In [ ]:
animar([0, 0, 0.2],  [0, 0, 0], q=-1, tmax=30, dt=1e-2, sobre_eje=True, rango=3, titulo="2.3")

### 2.4

In [ ]:
animar([0.01, 0, 0.2], [0, 0, 0], q=-1, tmax=10, sobre_eje=False, dist_stop=0.05, rango=3, titulo="2.4")

### Parte 3. Dispersion de una carga positiva

### 3.1 — sobre el eje y v0 = 1,2

In [ ]:
animar([0, 0, -4], [0, 0, 1.2], q=+1, tmax=10, sobre_eje=True, titulo="3.1 v0=1.2", rango=6)

### 3.2 — $x_0=0.5$, $v_0=1.2$

In [ ]:
animar([0.5, 0, -4], [0, 0, 1.2], q=+1, tmax=10, sobre_eje=False, titulo="3.2 v0=1.2", rango=6)

In [ ]:
animar([0.5, 0, -4], [0, 0.1, 1.2], q=+1, tmax=10, sobre_eje=False, titulo="3.3a", rango=6)

### 3.3 — $\vec v_0=(0,0.1,1.5)$

In [ ]:
animar([0.5, 0, -4], [0, 0.1, 1.5], q=+1, tmax=10, sobre_eje=False, titulo="3.3b", rango=6)

### 3.4 — Tabla de campo y $v_{\min}$

In [ ]:
# 3.4 Tabla de campo
print(f"{'punto':>16} | {'E_x':>10} | {'E_y':>10} | {'E_z':>10}")
for x in (0.0, 0.5):
    for z in (-0.5, -0.25):
        E = campo_electrico([x, 0, z])
        print(f"{str((x,0,z)):>16} | {E[0]:>10.5f} | {E[1]:>10.5f} | {E[2]:>10.5f}")

# Pregunta 3a
v_min = np.sqrt(2*(potencial_analitico(0) - potencial_analitico(-4))/M)
print("v_min =", v_min)